# UniLab Demo: 一键回放预训练策略 / 渲染 teaser 场景

`uv run demo <name>` 是 UniLab 的最简交互入口：

- 首次运行从 Hugging Face 自动拉取权重（5 个 checkpoint demo）或场景资源（teaser）。
- 缓存到本地后启动交互式仿真器或场景渲染器。
- 不需要训练，不需要先手动准备 checkpoint。

## 可用的 6 个 demo

| 名字 | 类型 | 内容 |
| --- | --- | --- |
| `dance` | checkpoint | G1 全身动作跟踪（dance motion） |
| `wallflip` | checkpoint | G1 后空翻 |
| `boxtracking` | checkpoint | G1 推箱子 |
| `locomani` | checkpoint | Go2 + 机械臂 操作-行走 |
| `inhandgrasp` | checkpoint | Sharpa 手内旋转抓取 |
| `teaser` | renderer | MotrixSim 论文封面场景静态展示 |

## 环境准备

如果你已经完成 README 里的 Quick Demo（`make setup-motrix` 或 `uv sync --extra motrix`），可以跳过这一节。

> **本地运行**：所有 demo 都会打开本地 GUI 窗口（MotrixSim viewer / MuJoCo viewer），**不支持 Google Colab**。

## Step 1: 确认环境

确保在 UniLab 项目根目录下运行本 notebook。

In [ ]:
from pathlib import Path

root = Path(".").resolve()
assert (root / "scripts").is_dir(), f"请在 UniLab 项目根目录运行此 notebook，当前目录: {root}"
assert (root / "conf").is_dir(), "缺少 conf/ 目录，请确认项目完整性"
print(f"项目根目录: {root}")

## Step 2: 查看 demo 注册表 + 本地缓存状态

checkpoint 缓存目录是 `src/unilab/assets/checkpoints/<demo名>/model_0.pt`。首次运行 `uv run demo <name>` 会自动从 Hugging Face dataset [`unilabsim/unilab-checkpoints`](https://huggingface.co/datasets/unilabsim/unilab-checkpoints) 拉取。teaser 走的是单独的 [`unilabsim/unilab-scenes`](https://huggingface.co/datasets/unilabsim/unilab-scenes) 仓库，由 `unilab.tools.render_teaser` 内部处理。

In [ ]:
from unilab.assets import ASSETS_ROOT_PATH
from unilab.demo import DEMO_REGISTRY

cache_dir = ASSETS_ROOT_PATH / "checkpoints"
for name in sorted(DEMO_REGISTRY):
    spec = DEMO_REGISTRY[name]
    if spec.entry == "teaser":
        status = "场景资源（首次运行自动从 unilab-scenes 拉取）"
    else:
        pt = cache_dir / name / "model_0.pt"
        status = "本地已缓存" if pt.is_file() else "未缓存（首次运行自动下载）"
    print(f"  {name:12s}  entry={spec.entry:18s}  {status}")

## Step 3: 运行一个 demo

下面的 cell 演示如何启动 `dance`。注意每个 demo 都会打开一个**交互式仿真器窗口**（4 个 motrix demo 走 MotrixSim viewer；`locomani` 走 MuJoCo viewer；`teaser` 走 MotrixSim 静态渲染）。在 notebook 里执行时 cell 会**阻塞**，直到你关闭窗口才会返回。

想看其他几个，把 `dance` 换成 `wallflip` / `boxtracking` / `locomani` / `inhandgrasp` / `teaser` 即可。

In [ ]:
!uv run demo dance

## 选项

- `--refresh` — 强制重新下载 checkpoint（先删除本地缓存再 resolve）。teaser 会忽略这个选项。
- `--device cpu` / `--device cuda:0` — 显式选择推理 device（仅 checkpoint demos 生效）。

```bash
uv run demo dance --refresh
uv run demo wallflip --device cpu
```

## Tab 补全

执行 `make setup-motrix` 后会自动在你的 shell rc 文件里注册补全脚本。新开 shell 后：

```text
uv run demo <TAB>            → boxtracking dance inhandgrasp locomani teaser wallflip
uv run demo d<TAB>           → dance
uv run demo dance --<TAB>    → --device --refresh
```

## 下一步

- 想训练自己的 checkpoint：`uv run train --algo ppo --task <task> --sim motrix`
- 想查看底层 dispatch 逻辑：[src/unilab/demo.py](../src/unilab/demo.py) 的 `DEMO_REGISTRY` 和 `build_demo_command`